# Organoids: embedding on site proportions

Within-gene site proportions are reduced to 50 principal components (seed 0), integrated across libraries with Harmony and embedded with UMAP (15 nearest neighbours on 10 components, seed 0). Used for Fig. 2A and figs. S2D, S3C and S3D.

## Locations

Paths are relative to the data directory given by the environment variable `M26_DATA_ROOT` (see the README); figure panels are written to `M26_RESULTS_ROOT`.

In [ ]:
# Locations (see README): M26_DATA_ROOT is required; M26_RESULTS_ROOT defaults to <repo>/results
import os
from pathlib import Path

REPO_ROOT = os.environ.get("M26_REPO_ROOT") or str(
    next(
        p
        for p in [Path.cwd(), *Path.cwd().parents]
        if (p / "environment" / "apa-python.yml").is_file()
    )
)
os.environ["M26_REPO_ROOT"] = REPO_ROOT
DATA_ROOT = os.environ["M26_DATA_ROOT"]
RESULTS_ROOT = os.environ.get("M26_RESULTS_ROOT", f"{REPO_ROOT}/results")
# Run in the directory this notebook originally ran in, so that relative paths resolve as they did
WORK_DIR = Path(DATA_ROOT, "notebooks/3UTR/organoids_knoblich/UTR")
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.chdir(WORK_DIR)

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import warnings

warnings.filterwarnings("ignore")

# ── Publication rcParams (Nature guidelines) ──────────────────────────
LAYOUT_DPI = 300
FONT_LABEL = 7
FONT_TICK = 6
FONT_LEGEND = 5.5
FONT_LEGEND_TITLE = 6
FONT_ANNOT = 6

plt.rcParams.update(
    {
        "figure.dpi": 150,
        "savefig.dpi": LAYOUT_DPI,
        "font.size": FONT_LABEL,
        "font.family": "sans-serif",
        "font.sans-serif": ["Arial", "Helvetica", "DejaVu Sans"],
        "axes.titlesize": FONT_LABEL,
        "axes.labelsize": FONT_LABEL,
        "axes.spines.top": False,
        "axes.spines.right": False,
        "pdf.fonttype": 42,
        "ps.fonttype": 42,
        "axes.linewidth": 0.8,
    }
)

sc.settings.verbosity = 0
print("Setup complete.")

In [ ]:
BASEPATH = f"{DATA_ROOT}/3utr/organoids/knoblich"
METADATA_PATH = f"{DATA_ROOT}/3utr/organoids/knoblich/metadata"

COUNTS_PATH = f"{BASEPATH}/cellranger"
COUNTS_FILE = f"{COUNTS_PATH}/aggregated_annotated_fix_filtered_normalized.h5ad"

UTRPATH = f"{BASEPATH}/utrcalling"

PEAKS_PROPORTIONS_FILENAME = (
    f"{UTRPATH}/merged_bw12_above10cts_apa_named_percentages_annotated_fix.h5ad"
)
RAW_UTR_COUNTS_FILENAME = f"{UTRPATH}/merged_raw_annotated_fix.h5ad"

RESULTS_DIR = Path(f"{BASEPATH}/utrcalling")
FIG_DIR = RESULTS_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

print(f"Results dir: {RESULTS_DIR}")
print(f"Figure dir: {FIG_DIR}")

# Load data


In [ ]:
adata_raw = sc.read_h5ad(COUNTS_FILE)
display(adata_raw)

In [ ]:
adata_utr_raw = sc.read_h5ad(RAW_UTR_COUNTS_FILENAME)
display(adata_utr_raw)

In [ ]:
adata_peaks_proportions = sc.read_h5ad(PEAKS_PROPORTIONS_FILENAME)
display(adata_peaks_proportions)

In [ ]:
peaks_df = adata_peaks_proportions.to_df()
sns.histplot(data=peaks_df.sum(axis=1), bins=50)

In [ ]:
# Transpose so peaks are rows, group by gene_name, sum peaks per gene → shape (n_genes, n_cells)
gene_sums = peaks_df.T.groupby(adata_peaks_proportions.var["gene_name"]).sum()
# Plot histogram of all values in gene_sums (flattened)
sns.histplot(data=gene_sums.values.flatten(), bins=50)
plt.yscale("symlog")

# Calculate UMAP for the peak-called UTR length dataset


In [ ]:
RANDOM_STATE = 0

In [ ]:
sc.tl.pca(adata_peaks_proportions, n_comps=50, random_state=RANDOM_STATE)

In [ ]:
adata_peaks_proportions.obs.head(2)

In [ ]:
sc.pl.pca_variance_ratio(adata_peaks_proportions, log=True)
sc.pl.pca(
    adata_peaks_proportions,
    color=["celltype", "batch_number", "nFeature_RNA"],
    ncols=1,
    title=["Cell type", "Batch", "Number of Features"],
)

In [ ]:
sc.pp.neighbors(adata_peaks_proportions, n_neighbors=15, n_pcs=10, random_state=RANDOM_STATE)
sc.tl.umap(adata_peaks_proportions, random_state=RANDOM_STATE)

In [ ]:
sc.pl.umap(
    adata_peaks_proportions,
    color=["celltype", "batch_number", "nFeature_RNA", "induced"],
    legend_fontsize=8,
    ncols=1,
    size=20,
    title=["Cell type", "Batch", "Number of Features", "Induced?"],
)

# Calculate UMAP for the raw UTR length dataset


In [ ]:
sc.tl.pca(adata_utr_raw, n_comps=50, random_state=RANDOM_STATE)

sc.pl.pca_variance_ratio(adata_utr_raw, log=True)
sc.pl.pca(
    adata_utr_raw,
    color=["celltype", "batch_number", "nFeature_RNA"],
    ncols=1,
    title=["Cell type", "Batch", "Number of Features"],
)
sc.pp.neighbors(adata_utr_raw, n_neighbors=50, n_pcs=8, random_state=RANDOM_STATE)
sc.tl.umap(adata_utr_raw, random_state=RANDOM_STATE)
sc.pl.umap(
    adata_utr_raw,
    color=["celltype", "batch_number", "nFeature_RNA"],
    legend_fontsize=8,
    ncols=1,
    size=20,
    title=["Cell type", "Batch", "Number of Features"],
)

# Batch-corrected UMAP for the peak-called UTR length dataset


In [ ]:
adata_peaks_proportions.var.head(2)

In [ ]:
# ── Pre-correction sanity check ───────────────────────────────────────────────
print(f"adata_peaks_proportions shape: {adata_peaks_proportions.shape}")

print("\nBatch composition (batch_number):")
print(adata_peaks_proportions.obs.groupby("batch_number").size().rename("n_cells").to_string())

print("\nCelltype composition:")
print(adata_peaks_proportions.obs.groupby("celltype").size().rename("n_cells").to_string())

# Convert batch_number to string/categorical in the original object so that
# colour palettes stay consistent across uncorrected, Harmony and Scanorama plots.
adata_peaks_proportions.obs["batch_number"] = (
    adata_peaks_proportions.obs["batch_number"].astype(str).astype("category")
)
print(
    "\nbatch_number dtype after cast:",
    adata_peaks_proportions.obs["batch_number"].dtype,
)

## 1. Harmony batch correction

Harmony (Korsunsky et al., 2019) iteratively adjusts the PCA embedding by finding shared cluster centroids across batches and projecting each cell towards the centroid that is most consistent across batches. The corrected embedding is stored in `X_pca_harmony` and used directly for kNN graph construction and UMAP layout.


In [ ]:
import harmonypy as hm
import scanpy.external as sce

# ── 1a. Harmony batch correction ──────────────────────────────────────────────
# We work on a deep copy to preserve the original (uncorrected) object.
adata_harmony = adata_peaks_proportions.copy()

# Harmony requires the batch key as a categorical string column.
adata_harmony.obs["batch_number"] = adata_harmony.obs["batch_number"].astype(str).astype("category")

# Re-run PCA; 50 components computed, 8 informative PCs used post-correction.
sc.tl.pca(adata_harmony, n_comps=50, random_state=RANDOM_STATE)

# ── Call harmonypy directly to handle v0.2.x (PyTorch) output layout ──────────
# The scanpy wrapper assumes Z_corr shape (n_pcs, n_cells) but harmonypy ≥0.2.0
# may store it as (n_cells, n_pcs) or as a PyTorch tensor; we normalise here.
_pca_mat = adata_harmony.obsm["X_pca"].astype(np.float64)

ho = hm.run_harmony(
    _pca_mat,
    adata_harmony.obs,
    "batch_number",
    max_iter_harmony=50,
    random_state=RANDOM_STATE,
)

# Convert to numpy regardless of whether Z_corr is a torch tensor or ndarray.
_Z = ho.Z_corr
if hasattr(_Z, "detach"):  # torch.Tensor path
    _Z = _Z.detach().cpu().numpy()
elif not isinstance(_Z, np.ndarray):
    _Z = np.array(_Z)

# Guarantee shape (n_cells, n_pcs); old harmonypy returned (n_pcs, n_cells).
if _Z.shape[0] != adata_harmony.n_obs:
    _Z = _Z.T

adata_harmony.obsm["X_pca_harmony"] = _Z

print("Harmony batch correction complete.")
print(f"  X_pca_harmony shape : {adata_harmony.obsm['X_pca_harmony'].shape}")

In [ ]:
# Build kNN graph on the Harmony-corrected embedding.
# n_pcs=10 is consistent with the uncorrected analysis.
sc.pp.neighbors(
    adata_harmony,
    n_neighbors=15,
    use_rep="X_pca_harmony",
    n_pcs=10,
    random_state=RANDOM_STATE,
)

sc.tl.umap(adata_harmony, random_state=RANDOM_STATE)

print(f"  X_umap shape        : {adata_harmony.obsm['X_umap'].shape}")

In [ ]:
# ── 1b. Harmony - PCA inspection (before/after) ───────────────────────────────
# Overlay corrected PCA to verify that batch-specific clusters have collapsed.
sc.pl.pca(
    adata_harmony,
    color=["celltype", "batch_number"],
    ncols=2,
    title=["Harmony PCA — cell type", "Harmony PCA — batch"],
    size=20,
    wspace=0.4,
    show=False,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "harmony_pca_inspection.pdf", bbox_inches="tight")
plt.show()

In [ ]:
# ── 1c. Harmony - UMAP overview ───────────────────────────────────────────────
sc.pl.umap(
    adata_harmony,
    color=["celltype", "batch_number", "sample_id", "nFeature_RNA", "induced"],
    legend_fontsize=FONT_LEGEND,
    ncols=2,
    size=20,
    wspace=0.3,
    title=["Cell type", "Batch", "Sample", "# RNA features", "Induced?"],
    show=False,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "harmony_umap_overview.pdf", bbox_inches="tight")
plt.show()

## 2. Scanorama batch correction

Scanorama (Hie et al., 2019) identifies mutual nearest-neighbour (MNN) pairs across all pairs of batches in PCA space and applies a linear correction derived from those pairs. The corrected PCA-space embedding is stored in `X_pca_scanorama`. Because Scanorama compresses the joint embedding into the full 50 PCA dimensions, we use all components for the downstream kNN graph.


In [ ]:
# ── 2a. Scanorama batch correction ────────────────────────────────────────────
adata_scanorama = adata_peaks_proportions.copy()

# Scanorama requires the batch key as a categorical string column.
adata_scanorama.obs["batch_number"] = (
    adata_scanorama.obs["batch_number"].astype(str).astype("category")
)

# Scanorama requires cells to be contiguous by batch.
# Sort, integrate, then restore the original cell order.
_orig_order = adata_scanorama.obs_names.copy()
adata_scanorama = adata_scanorama[adata_scanorama.obs.sort_values("batch_number").index].copy()

# (Re-)run PCA on the sorted copy.
sc.tl.pca(adata_scanorama, n_comps=50, random_state=RANDOM_STATE)

# Scanorama integration: corrects X_pca in PCA space → X_pca_scanorama.
# The corrected embedding retains 50 dimensions (= min(dimred=100, n_comps=50)).
sce.pp.scanorama_integrate(
    adata_scanorama,
    key="batch_number",
    basis="X_pca",
    adjusted_basis="X_pca_scanorama",
)

# Restore original cell order so that adata_scanorama aligns with other objects.
adata_scanorama = adata_scanorama[_orig_order].copy()

# Build kNN graph.
# Unlike PCA, the Scanorama embedding dimensions are not ordered by variance;
# we therefore run a compact PCA on the corrected embedding first, so the
# downstream neighbour graph uses a variance-ranked space - matching the
# strategy used for the uncorrected and Harmony embeddings.
sc.tl.pca(
    adata_scanorama,
    n_comps=50,
    use_highly_variable=False,
    random_state=RANDOM_STATE,
    # operate on the Scanorama embedding, not on X
)

# Note: sc.tl.pca operates on .X.  We want PCA on the Scanorama embedding,
# so we use sklearn directly.
from sklearn.decomposition import PCA as _PCA

_pca_scan = _PCA(n_components=50, random_state=RANDOM_STATE)
adata_scanorama.obsm["X_pca_scanorama_pca"] = _pca_scan.fit_transform(
    adata_scanorama.obsm["X_pca_scanorama"]
)

print("Scanorama batch correction complete.")
print(f"  X_pca_scanorama shape     : {adata_scanorama.obsm['X_pca_scanorama'].shape}")
print(f"  X_pca_scanorama_pca shape : {adata_scanorama.obsm['X_pca_scanorama_pca'].shape}")

In [ ]:
sc.pp.neighbors(
    adata_scanorama,
    n_neighbors=50,
    use_rep="X_pca_scanorama_pca",
    n_pcs=10,
    random_state=RANDOM_STATE,
)

sc.tl.umap(adata_scanorama, random_state=RANDOM_STATE)
print(f"  X_umap shape              : {adata_scanorama.obsm['X_umap'].shape}")

In [ ]:
# ── 2b. Scanorama - UMAP overview ────────────────────────────────────────────
sc.pl.umap(
    adata_scanorama,
    color=["celltype", "batch_number", "sample_id", "nFeature_RNA"],
    legend_fontsize=FONT_LEGEND,
    ncols=2,
    size=20,
    wspace=0.3,
    title=["Cell type", "Batch", "Sample", "# RNA features"],
    show=False,
)
plt.tight_layout()
plt.savefig(FIG_DIR / "scanorama_umap_overview.pdf", bbox_inches="tight")
plt.show()

## 3. Side-by-side comparison: uncorrected · Harmony · Scanorama

The figure below shows the three UMAP layouts in a single panel grid, coloured by cell type (left column) and batch (right column). A successful batch correction should produce interleaved batch colours that follow the same cell-type structure as the uncorrected layout.


In [ ]:
# ── 3. Comparison figure ─────────────────────────────────────────
# Layout: 3 rows (Uncorrected / Harmony / Scanorama) × 2 cols (cell type / batch).
# Dimensions chosen to fit within a two-column figure (≤ 7.2 in) or as a
# single-column panel (≤ 3.5 in) after final journal resizing.

_panel_w, _panel_h = 4, 2.0
_ncols, _nrows = 2, 3

fig, axes = plt.subplots(
    _nrows,
    _ncols,
    figsize=(_panel_w * _ncols, _panel_h * _nrows),
)
fig.subplots_adjust(hspace=0.45, wspace=0.7)

_methods = [
    ("Uncorrected", adata_peaks_proportions),
    ("Harmony", adata_harmony),
    ("Scanorama", adata_scanorama),
]
_col_keys = ["celltype", "batch_number"]
_col_titles = ["Cell type", "Batch"]

for r, (label, adt) in enumerate(_methods):
    for c, (col_key, col_title) in enumerate(zip(_col_keys, _col_titles)):
        ax = axes[r, c]

        sc.pl.umap(
            adt,
            color=col_key,
            ax=ax,
            show=False,
            legend_loc="right margin",
            legend_fontsize=FONT_LEGEND,
            size=6,
            title="",
            frameon=False,
        )

        # Column header (top row only)
        if r == 0:
            ax.set_title(col_title, fontsize=FONT_LABEL, fontweight="bold", pad=4)

        # Row label on the leftmost panel only
        if c == 0:
            ax.set_ylabel(
                f"{label}\nUMAP 2",
                fontsize=FONT_TICK,
                labelpad=2,
                fontweight="bold" if True else "normal",
            )
        else:
            ax.set_ylabel("")

        ax.set_xlabel("UMAP 1", fontsize=FONT_TICK, labelpad=1)

plt.savefig(
    FIG_DIR / "umap_batch_correction_comparison.pdf",
    bbox_inches="tight",
    dpi=LAYOUT_DPI,
)
plt.savefig(
    FIG_DIR / "umap_batch_correction_comparison.png",
    bbox_inches="tight",
    dpi=LAYOUT_DPI,
)
plt.show()
print(f"Comparison figure saved to {FIG_DIR}")

In [ ]:
# ── 4. Save Harmony corrected AnnData object ─────────────────────────────────────────
# Both objects carry the corrected PCA embedding, the kNN graph, and the UMAP
# coordinates, making them ready for downstream compositional analyses
# (Dirichlet-Multinomial regression, pseudotime, etc.).

_base = "merged_bw12_above10cts_apa_named_percentages_annotated"
_harmony_path = RESULTS_DIR / f"{_base}_harmony.h5ad"
# _scanorama_path = RESULTS_DIR / f"{_base}_scanorama.h5ad"

adata_harmony.write_h5ad(_harmony_path, compression="gzip")
# adata_scanorama.write_h5ad(_scanorama_path, compression="gzip")

print(f"Harmony   → {_harmony_path}")
# print(f"Scanorama → {_scanorama_path}")